# 🎓 Day 4 — Large Language Model (Mistral) | Anand Kumar
## AI-Powered Student Performance & Dropout Predictor
### FDP 5-Day Academic AI Project

**Goal:** Use HuggingFace Inference API (Mistral-7B-Instruct) to generate personalised faculty counselling letters using outputs from Days 1–3 as context.

**Output:** `generate_counsellor_letter(name, semester, risk, cgpa, nlp)` → `{'letter': 'Dear Rahul...'}`

---
**Primary model:** `mistralai/Mistral-7B-Instruct-v0.2` via HF Inference API (free tier)  
**Fallback 1:** Google Gemini 1.5 Flash API  
**Fallback 2:** `google/flan-t5-large` running directly in Colab (no API key needed)  
**Fallback 3:** Rule-based letter generator  
**Note:** Day 4 uses the API — no model training. T4 GPU used only for Flan-T5 fallback.

---
### 🔑 Before You Start:
1. Go to [hf.co/settings/tokens](https://hf.co/settings/tokens) → **New token** (Read permission)
2. Copy the token and paste it in Cell 4 where it says `hf_xxxx`
3. (Optional) Get a free Gemini key from [console.cloud.google.com](https://console.cloud.google.com)


In [23]:
# Cell 1 — Install
!pip install -q huggingface_hub openai google-genai
# Note: google-genai is the NEW SDK (replaces deprecated google-generativeai)
# Note: openai package is used as OpenAI-compatible client for HF router

In [ ]:
# ── OPTION A: HuggingFace Inference Providers ──────────────────────────
# Uses HF's router which connects to real GPU providers (Cerebras, SambaNova etc.)
# All these models ARE hosted with active inference — verified working April 2025
from openai import OpenAI
HF_TOKEN = "your_hf_token"   # hf.co/settings/tokens → New token → Read
# HF router is OpenAI-compatible — just change base_url and api_key
hf_client = OpenAI(
base_url="https://router.huggingface.co/v1",
api_key=HF_TOKEN,
)
# ── Verified working models on HF Inference Providers (free tier) ──────
# Pick ONE — try top to bottom, use whichever responds fastest for you
HF_MODELS = [
"Qwen/Qwen2.5-7B-Instruct",
# Best choice — fast, free, strong
"meta-llama/Llama-3.1-8B-Instruct",   # Very reliable, widely hosted
"google/gemma-2-2b-it",
# Lightest model, fastest response
"mistralai/Mistral-7B-Instruct-v0.3", # v0.3 has inference — v0.2 does not
]
HF_MODEL = HF_MODELS[0]   # start with Qwen2.5-7B — change index if needed
print(f"Using HF model: {HF_MODEL}")
# Quick test
test = hf_client.chat.completions.create(
model=HF_MODEL,
messages=[{"role": "user", "content": "Say: HF working"}],
max_tokens=20,
)
print("HF test:", test.choices[0].message.content)

Using HF model: Qwen/Qwen2.5-7B-Instruct
HF test: HF working


In [31]:
# ── OPTION B: Google Gemini 2.5 Flash (Fallback) ───────────────────────
# Step 1: Get API key → go to aistudio.google.com → "Get API key" → Create → Copy
# Step 2: In Colab left panel → click key icon (Secrets) → Add new secret:
# Name: GEMINI_API_KEY   Value: paste your key → toggle "Notebook access" ON
# IMPORTANT: google-generativeai is DEPRECATED as of Nov 2025
# Use the new 'google-genai' package instead
from google.colab import userdata
from google import genai as google_genai
GEMINI_KEY = userdata.get('GEMINI_API_KEY')   # reads from Colab Secrets safely
gemini_client = google_genai.Client(api_key=GEMINI_KEY)
GEMINI_MODEL  = "gemini-2.5-flash"
# Quick test
# current free tier model
test_gemini = gemini_client.models.generate_content(model=GEMINI_MODEL,contents="Say: Gemini working")
print("Gemini test:", test_gemini.text)

Gemini test: Gemini working


In [32]:
# ── System prompt ─────────────────────────────────────────────────────
SYSTEM = """You are an experienced academic counsellor at an engineering college.
Write a formal, empathetic, and constructive counselling letter to a student.
Be specific — reference their actual data. Do not use generic phrases.
Keep the letter to 4 sentences. End with exactly 2 concrete action items."""
def build_prompt(name, semester, risk, cgpa_result, nlp_result):
    kw = ', '.join(nlp_result['keywords']) if nlp_result['keywords'] else 'various concerns'
    return f"""Student Name      : {name}
Current Semester  : Semester {semester}
Dropout Risk      : {risk['risk_label']} (confidence {risk['risk_score']:.0%})
Predicted CGPA    : {cgpa_result['predicted_cgpa']} | Trend: {cgpa_result['trend']}
Concern Category  : {nlp_result['concern']}  |  Urgency: {nlp_result['urgency']}
Key Issues        : {kw}

Write a formal 4-sentence counselling letter for this student.
Address the student directly as 'Dear {name},'.
End with exactly 2 specific action items for this week."""

In [35]:
# ── Caller: Option A — HuggingFace Router ────────────────────────────
def call_hf(prompt):
    response = hf_client.chat.completions.create(
        model=HF_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": prompt}
        ],
        max_tokens=300,
        temperature=0.4,
    )
    return response.choices[0].message.content.strip()


# ── Caller: Option B — Gemini (new SDK) ──────────────────────────────
def call_gemini(prompt):
    full_prompt = SYSTEM + "\n\n" + prompt
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=full_prompt,
        config=google_genai.types.GenerateContentConfig(
            temperature=0.4,
            max_output_tokens=300,
        )
    )
    return response.text.strip()


# ── Caller: Option C — Flan-T5 (already working in your Colab) ───────
# You said Flan is already working — keep as-is
def call_flan(prompt):
    # assumes 'flan' pipeline is already loaded from previous session
    out = flan(SYSTEM[:200] + " " + prompt[:400], max_new_tokens=200)
    return out[0]['generated_text'].strip()


# ── Rule-based fallback (always works, no API needed) ─────────────────
def rule_based_fallback(name, risk, cgpa_result, nlp_result):
    return (
        f"Dear {name}, your academic record indicates a {risk['risk_label']} dropout risk "
        f"with a predicted CGPA of {cgpa_result['predicted_cgpa']} showing a {cgpa_result['trend']} trend."
        f"Your primary concern has been classified as {nlp_result['concern']} with "
        f"{nlp_result['urgency']} urgency — immediate intervention is recommended. "
        f"Action 1: Visit the academic counselling office within 48 hours. "
        f"Action 2: Submit all pending assignments by end of this week."
    )


# ── Master function: generate_counsellor_letter() ─────────────────────
def generate_counsellor_letter(name, semester, risk, cgpa_result, nlp_result):
    prompt = build_prompt(name, semester, risk, cgpa_result, nlp_result)

    # Try Option A first — HuggingFace
    try:
        print("Trying HF Inference Provider...")
        letter = call_hf(prompt)
        print("HF succeeded.")
        return {'letter': letter, 'source': 'HuggingFace'}

    except Exception as e1:
        print(f"HF failed: {e1}")

        # Try Option B — Gemini
        try:
            print("Trying Gemini...")
            letter = call_gemini(prompt)
            print("Gemini succeeded.")
            return {'letter': letter, 'source': 'Gemini'}

        except Exception as e2:
            print(f"Gemini failed: {e2}")

            # Try Option C — Flan-T5
            try:
                print("Trying Flan-T5...")
                letter = call_flan(prompt)
                print("Flan-T5 succeeded.")
                return {'letter': letter, 'source': 'Flan-T5'}

            except Exception as e3:
                print(f"Flan-T5 failed: {e3}. Using rule-based fallback.")
                letter = rule_based_fallback(name, risk, cgpa_result, nlp_result)
                return {'letter': letter, 'source': 'Rule-based'}

In [36]:
# ── Test each option individually first ───────────────────────────────
SAMPLE = {
    'name': 'Rahul Kumar', 'semester': 4,
    'risk':  {'risk_label': 'HIGH',    'risk_score': 0.82},
    'cgpa':  {'predicted_cgpa': 5.6,   'trend': 'DECLINING'},
    'nlp':   {'concern': 'ACADEMIC',   'urgency': 'IMMEDIATE',
               'keywords': ['backlog', 'attendance', 'exam']}
}

# Test HF alone
print("=== Testing HF Option A ===")
try:
    r = call_hf(build_prompt(**{k:v for k,v in SAMPLE.items() if k != 'risk' and k != 'cgpa' and k != 'nlp'},
                              risk=SAMPLE['risk'], cgpa_result=SAMPLE['cgpa'], nlp_result=SAMPLE['nlp']))
    print(r[:200])
except Exception as e:
    print("HF Error:", e)

# Test Gemini alone
print("\n=== Testing Gemini Option B ===")
try:
    r = call_gemini(build_prompt('Rahul Kumar', 4, SAMPLE['risk'], SAMPLE['cgpa'], SAMPLE['nlp']))
    print(r[:200])
except Exception as e:
    print("Gemini Error:", e)

# Test full fallback chain with 3 different student profiles
print("\n=== Full chain test — 3 profiles ===")
profiles = [
    ('Rahul Kumar',   4, {'risk_label':'HIGH',   'risk_score':0.82},
                         {'predicted_cgpa':5.6,  'trend':'DECLINING'},
                         {'concern':'ACADEMIC',  'urgency':'IMMEDIATE', 'keywords':['backlog','attendance']}),
    ('Priya Sharma',  3, {'risk_label':'MEDIUM', 'risk_score':0.61},
                         {'predicted_cgpa':6.4,  'trend':'STABLE'},
                         {'concern':'FINANCIAL', 'urgency':'MODERATE', 'keywords':['fee','scholarship']}),
    ('Amit Singh',    5, {'risk_label':'HIGH',   'risk_score':0.91},
                         {'predicted_cgpa':4.8,  'trend':'DECLINING'},
                         {'concern':'MENTAL_HEALTH','urgency':'IMMEDIATE','keywords':['stress','hopeless']}),
]

for name, sem, risk, cgpa, nlp in profiles:
    result = generate_counsellor_letter(name, sem, risk, cgpa, nlp)
    print(f"\n--- {name} (via {result['source']}) ---")
    print(result['letter'])
    print("-" * 60)

=== Testing HF Option A ===
Dear Rahul Kumar,

Given your current academic performance, including the backlog, declining trend in your CGPA, and low attendance, we are concerned about your risk of dropping out, which has been as

=== Testing Gemini Option B ===
Gemini Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

=== Full chain test — 3 profiles ===
Trying HF Inference Provider...
HF succeeded.

--- Rahul Kumar (via HuggingFace) ---
Dear Rahul Kumar,

Given the high dropout risk and the declining trend in your predicted CGPA, we are deeply concerned about your current academic performance. Your backlog and poor attendance are significant contributors to this situation. To address these issues, we urge you to schedule a meeting with your academic advisor this week to discuss a personalized study plan and to improve your attendance. 

In [37]:
# ── Save final llm_module.py to Google Drive ──────────────────────────
llm_module_code = '''
import os
from openai import OpenAI
from google import genai as google_genai
from google.colab import userdata
# ── Config ─────────────────────────────────────────────────────────────
HF_TOKEN    = os.getenv('HF_TOKEN', 'hf_xxxxxxxxxxxxxxxxxxxx')
GEMINI_KEY  = os.getenv('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY'))
HF_MODEL    = "Qwen/Qwen2.5-7B-Instruct"
GEMINI_MODEL= "gemini-2.5-flash"
hf_client     = OpenAI(base_url="https://router.huggingface.co/v1", api_key=HF_TOKEN)
gemini_client  = google_genai.Client(api_key=GEMINI_KEY)
SYSTEM = """You are an academic counsellor. Write a formal 4-sentence counselling
letter. Reference actual student data. End with 2 specific action items."""
def build_prompt(name, semester, risk, cgpa_result, nlp_result):
kw = ", ".join(nlp_result.get("keywords", []))
return f"""Student: {name} | Semester: {semester}
Risk: {risk["risk_label"]} ({risk["risk_score"]:.0%}) | CGPA: {cgpa_result["predicted_cgpa"]}
({cgpa_result["trend"]})
Concern: {nlp_result["concern"]} | Urgency: {nlp_result["urgency"]} | Issues: {kw}
Write a formal 4-sentence counselling letter. End with 2 action items."""
def generate_counsellor_letter(name, semester, risk, cgpa_result, nlp_result):
prompt = build_prompt(name, semester, risk, cgpa_result, nlp_result)
try:
r = hf_client.chat.completions.create(
model=HF_MODEL,
messages=[{"role":"system","content":SYSTEM},{"role":"user","content":prompt}],
max_tokens=300, temperature=0.4)
return {"letter": r.choices[0].message.content.strip(), "source": "HuggingFace"}
except Exception:
        pass
    try:
        r = gemini_client.models.generate_content(
            model=GEMINI_MODEL, contents=SYSTEM+"\\n\\n"+prompt,
            config=google_genai.types.GenerateContentConfig(temperature=0.4,
max_output_tokens=300))
        return {"letter": r.text.strip(), "source": "Gemini"}
    except Exception:
        pass
    return {"letter": (f"Dear {name}, your academic record shows {risk['risk_label']} dropout risk "
                       f"with predicted CGPA {cgpa_result['predicted_cgpa']} ({cgpa_result['trend']}). "
                       f"Concern: {nlp_result['concern']} ({nlp_result['urgency']}). "
                       f"Action 1: Visit counselling office. Action 2: Clear all backlogs this week."),
            "source": "Rule-based"}
'''

with open(f'{SAVE_DIR}/llm_module.py', 'w') as f:
    f.write(llm_module_code)
print("Saved: llm_module.py to Drive")

Saved: llm_module.py to Drive
